# 📊 Análisis de Sales Dataset con Pandas
## Dataset Normalizado — Exploración, Filtros y Visualizaciones

> **El análisis cubre:**
> - Exploración general del dataset (EDA)
> - Ventas por país, año, producto y cliente
> - Filtros por estado, deal size y territorio
> - Tablas pivot cruzadas
> - Gráficos con matplotlib

---
### 👨‍🏫 Stack tecnológico
| Componente | Herramienta |
|---|---|
| Data wrangling | **Pandas** |
| Visualizaciones | **Matplotlib + Seaborn** |
| Tablas formateadas | **Tabulate** |

---
## 🗺️ Ruta de Trabajo

```
FASE 1 — SETUP
  ├── Paso 1 │ Instalar dependencias
  └── Paso 2 │ Importar librerías

FASE 2 — DATOS
  ├── Paso 3 │ Cargar dataset
  └── Paso 4 │ Exploración EDA

FASE 3 — COLUMNAS DERIVADAS
  └── Paso 5 │ Enriquecer dataset

FASE 4 — ANÁLISIS
  ├── Paso 6 │ Ventas por año y país
  ├── Paso 7 │ Análisis por producto
  ├── Paso 8 │ Análisis por cliente
  ├── Paso 9 │ Tablas pivot
  └── Paso 10│ Gráficos
```


## 📦 PASO 1 — Instalar Dependencias

In [ ]:
# ─── Instalar dependencias ───────────────────────────────────────
print("⏳ Instalando dependencias...")

!pip install -q pandas matplotlib seaborn tabulate openpyxl

print("\n✅ Dependencias instaladas correctamente")


## 🐍 PASO 2 — Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from tabulate import tabulate
from IPython.display import display

# Estilo de gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Librerías importadas correctamente")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")
print(f"   matplotlib : {plt.matplotlib.__version__}")


## 🗃️ PASO 3 — Cargar el Dataset

### Subir el archivo a Colab:
- Ejecutá la celda → aparece un botón **"Elegir archivos"**
- Seleccioná `sales_data_normalized.csv` desde tu PC


In [ ]:
# ─── Subir el CSV desde tu PC ────────────────────────────────────
from google.colab import files

print("📂 Seleccioná el archivo sales_data_normalized.csv...")
uploaded = files.upload()

# ─── Cargar el CSV ───────────────────────────────────────────────
CSV_PATH = 'sales_data_normalized.csv'

df = pd.read_csv(CSV_PATH, parse_dates=['orderdate'])

print(f"\n✅ Dataset cargado correctamente")
print(f"   Filas    : {df.shape[0]:,}")
print(f"   Columnas : {df.shape[1]}")
print(f"   Período  : {df['orderdate'].min().date()} → {df['orderdate'].max().date()}")
print(f"\n📋 Columnas disponibles:")
print(df.columns.tolist())


## 🔍 PASO 4 — Exploración General (EDA)

In [ ]:
# ── 1. Tipos de datos y nulos ─────────────────────────────────────
print("=" * 60)
print("📋 TIPOS DE DATOS Y NULOS")
print("=" * 60)
info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'No Nulos': df.count(),
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(1)
})
print(tabulate(info_df, headers='keys', tablefmt='grid'))


In [ ]:
# ── 2. Estadísticas numéricas ─────────────────────────────────────
print("=" * 60)
print("📈 ESTADÍSTICAS NUMÉRICAS CLAVE")
print("=" * 60)
cols_num = ['quantityordered', 'priceeach', 'sales', 'msrp', 'dealsize_num']
display(df[cols_num].describe().round(2))


In [ ]:
# ── 3. Distribución de variables categóricas ──────────────────────
print("=" * 60)
print("🏷️  VARIABLES CATEGÓRICAS")
print("=" * 60)

for col in ['status', 'productline', 'dealsize', 'territory']:
    vc = df[col].value_counts()
    print(f"\n🔹 {col}:")
    for val, cnt in vc.items():
        bar = '█' * int(cnt / 50)
        print(f"   {str(val):<25} {cnt:>4}  {bar}")


In [ ]:
# ── 4. Top 10 países por número de órdenes ────────────────────────
print("=" * 60)
print("🌎 TOP 10 PAÍSES POR NÚMERO DE ÓRDENES")
print("=" * 60)
country_top = df['country'].value_counts().head(10)
for country, cnt in country_top.items():
    bar = '█' * int(cnt / 20)
    print(f"   {country:<20} {cnt:>4}  {bar}")


## 🔧 PASO 5 — Enriquecer con Columnas Derivadas

Agregamos columnas calculadas para facilitar el análisis:
- `order_year`, `order_month`, `order_quarter` — desde la fecha
- `revenue_calc` — cantidad × precio
- `discount_pct` — descuento respecto al MSRP
- `contact_full` — nombre completo del contacto
- `full_address` — dirección en una sola línea


In [ ]:
# ═══ COLUMNAS DERIVADAS ══════════════════════════════════════════

# Derivadas de fecha
df['order_year']    = df['orderdate'].dt.year.astype('Int64')
df['order_month']   = df['orderdate'].dt.month.astype('Int64')
df['order_quarter'] = df['orderdate'].dt.quarter.astype('Int64')

# Derivadas de negocio
df['revenue_calc'] = (df['quantityordered'] * df['priceeach']).round(2)
df['discount_pct'] = ((1 - df['priceeach'] / df['msrp']) * 100).clip(0).round(1)
df['contact_full'] = (
    df['contactfirstname'].fillna('') + ' ' + df['contactlastname'].fillna('')
).str.strip()

# Dirección completa
df['addressline2_clean'] = df['addressline2'].apply(
    lambda x: f', {x}' if pd.notna(x) and str(x).strip() not in ['', 'nan'] else ''
)
df['full_address'] = (
    df['addressline1'].fillna('').str.strip()
    + df['addressline2_clean']
    + ', ' + df['city'].fillna('')
    + df['state'].apply(lambda x: f', {x}' if pd.notna(x) and str(x) not in ['N/A', 'nan'] else '')
    + ', ' + df['country'].fillna('')
)
df.drop(columns=['addressline2_clean'], inplace=True)
df['ordernumber'] = df['ordernumber'].astype(str)

print("✅ Columnas derivadas agregadas:")
nuevas = ['order_year', 'order_month', 'order_quarter',
          'revenue_calc', 'discount_pct', 'contact_full', 'full_address']
for c in nuevas:
    print(f"   ✚ {c}")

print(f"\n   Total columnas ahora: {len(df.columns)}")
print(f"\n🏠 Ejemplo full_address: {df['full_address'].iloc[0]}")


## 📅 PASO 6 — Ventas por Año y País

In [ ]:
# ── Ventas totales por año ────────────────────────────────────────
print("📅 VENTAS TOTALES POR AÑO")
yr = df.groupby('order_year')['sales'].agg(
    Total_USD=('sum'),
    N_Ordenes=('count'),
    Promedio_USD=('mean')
).round(2)
yr.columns = ['Total $', 'N Órdenes', 'Promedio $']
print(tabulate(yr, headers='keys', tablefmt='grid'))


In [ ]:
# ── Top 10 países por ventas totales ─────────────────────────────
print("\n🌎 TOP 10 PAÍSES POR VENTAS TOTALES")
top_pais = (
    df.groupby('country')['sales']
    .agg(Total=('sum'), Ordenes=('count'), Promedio=('mean'))
    .round(2)
    .sort_values('Total', ascending=False)
    .head(10)
)
top_pais.columns = ['Total $', 'N Órdenes', 'Promedio $']
print(tabulate(top_pais, headers='keys', tablefmt='grid'))


In [ ]:
# ── Ventas por país Y año (tabla cruzada) ────────────────────────
print("\n📊 VENTAS POR PAÍS Y AÑO (Top 8 países)")
top8_paises = df.groupby('country')['sales'].sum().nlargest(8).index
subset = df[df['country'].isin(top8_paises)]
tabla = subset.pivot_table(
    values='sales', index='country',
    columns='order_year', aggfunc='sum', fill_value=0
).round(0).astype(int)
tabla['TOTAL'] = tabla.sum(axis=1)
tabla = tabla.sort_values('TOTAL', ascending=False)
print(tabulate(tabla, headers='keys', tablefmt='grid'))


In [ ]:
# ── Ventas por trimestre ─────────────────────────────────────────
print("\n📆 VENTAS POR TRIMESTRE (todos los años)")
qtr = df.groupby(['order_year', 'order_quarter'])['sales'].sum().round(2).reset_index()
qtr.columns = ['Año', 'Trimestre', 'Total $']
qtr['Trimestre'] = 'Q' + qtr['Trimestre'].astype(str)
display(qtr.pivot(index='Año', columns='Trimestre', values='Total $').fillna(0).astype(int))


## 🚗 PASO 7 — Análisis por Producto

In [ ]:
# ── Ventas por línea de producto ─────────────────────────────────
print("🚗 VENTAS POR LÍNEA DE PRODUCTO")
prod = df.groupby('productline').agg(
    Total_USD=('sales', 'sum'),
    N_Ordenes=('ordernumber', 'count'),
    Precio_Prom=('priceeach', 'mean'),
    Descuento_Prom=('discount_pct', 'mean'),
    MSRP_Prom=('msrp', 'mean'),
    DealSize_Prom=('dealsize_num', 'mean')
).round(2).sort_values('Total_USD', ascending=False)
prod.columns = ['Total $', 'N Órdenes', 'Precio Prom $', 'Descuento %', 'MSRP Prom $', 'Deal Size Prom']
print(tabulate(prod, headers='keys', tablefmt='grid'))


In [ ]:
# ── Top 15 productos más vendidos (por código) ───────────────────
print("\n🏆 TOP 15 PRODUCTOS MÁS VENDIDOS (por código)")
top_prod = (
    df.groupby(['productcode', 'productline'])['sales']
    .sum().round(2)
    .reset_index()
    .sort_values('sales', ascending=False)
    .head(15)
)
top_prod.columns = ['Código', 'Línea', 'Total $']
print(tabulate(top_prod, headers='keys', tablefmt='grid', showindex=False))


In [ ]:
# ── Deal size por línea de producto ──────────────────────────────
print("\n📦 DISTRIBUCIÓN DE DEAL SIZE POR LÍNEA DE PRODUCTO")
deal = df.pivot_table(
    values='ordernumber', index='productline',
    columns='dealsize', aggfunc='count', fill_value=0
)
deal['TOTAL'] = deal.sum(axis=1)
print(tabulate(deal, headers='keys', tablefmt='grid'))


## 👥 PASO 8 — Análisis por Cliente

In [ ]:
# ── Top 15 clientes por ventas totales ───────────────────────────
print("🏆 TOP 15 CLIENTES POR VENTAS TOTALES")
top_clientes = (
    df.groupby(['customername', 'country', 'city']).agg(
        Total_USD=('sales', 'sum'),
        N_Ordenes=('ordernumber', 'count'),
        Productos=('productline', lambda x: ', '.join(sorted(x.unique())))
    ).round(2)
    .sort_values('Total_USD', ascending=False)
    .head(15)
    .reset_index()
)
top_clientes.columns = ['Cliente', 'País', 'Ciudad', 'Total $', 'N Órdenes', 'Productos']
print(tabulate(top_clientes, headers='keys', tablefmt='grid', showindex=False))


In [ ]:
# ── Clientes por estado de órdenes ───────────────────────────────
print("\n📋 ÓRDENES POR ESTADO")
status_df = df.groupby('status').agg(
    N_Ordenes=('ordernumber', 'count'),
    Total_USD=('sales', 'sum'),
    Prom_USD=('sales', 'mean'),
    N_Paises=('country', 'nunique'),
    N_Clientes=('customername', 'nunique')
).round(2).sort_values('Total_USD', ascending=False)
status_df.columns = ['N Órdenes', 'Total $', 'Promedio $', 'Países', 'Clientes']
print(tabulate(status_df, headers='keys', tablefmt='grid'))


In [ ]:
# ── Función: buscar cliente por nombre ───────────────────────────
def buscar_cliente(nombre: str):
    """Filtrá todas las órdenes de un cliente por nombre (parcial, sin importar mayúsculas)."""
    subset = df[df['customername'].str.contains(nombre, case=False, na=False)]
    if subset.empty:
        print(f"❌ No se encontraron órdenes para: '{nombre}'")
        return

    print(f"\n👤 CLIENTE: {nombre.upper()}")
    print(f"   Dirección  : {subset['full_address'].iloc[0]}")
    print(f"   Contacto   : {subset['contact_full'].iloc[0]} | Tel: {subset['phone'].iloc[0]}")
    print(f"   N Órdenes  : {len(subset)}")
    print(f"   Total $    : ${subset['sales'].sum():,.2f}")
    print(f"   Promedio $ : ${subset['sales'].mean():,.2f}")
    print(f"\n📋 Órdenes:")
    cols = ['ordernumber', 'orderdate', 'productline', 'productcode',
            'sales', 'status', 'dealsize']
    display(subset[cols].sort_values('orderdate', ascending=False))

# Ejemplo de uso:
buscar_cliente("Mini Gifts")


## 📊 PASO 9 — Tablas Pivot

In [ ]:
# ── Pivot: ventas por país y línea de producto ───────────────────
print("📊 VENTAS POR PAÍS Y LÍNEA DE PRODUCTO (Top 12 países)")
top12 = df.groupby('country')['sales'].sum().nlargest(12).index
pivot_pais_prod = df[df['country'].isin(top12)].pivot_table(
    values='sales', index='country',
    columns='productline', aggfunc='sum', fill_value=0
).round(0).astype(int)
pivot_pais_prod['TOTAL'] = pivot_pais_prod.sum(axis=1)
pivot_pais_prod = pivot_pais_prod.sort_values('TOTAL', ascending=False)
print(tabulate(pivot_pais_prod, headers='keys', tablefmt='grid'))


In [ ]:
# ── Pivot: deal size por territorio ──────────────────────────────
print("\n🌎 DEAL SIZE PROMEDIO POR TERRITORIO")
terr = df.groupby('territory').agg(
    N_Ordenes=('ordernumber', 'count'),
    Total_USD=('sales', 'sum'),
    DealSize_Prom=('dealsize_num', 'mean'),
    Descuento_Prom=('discount_pct', 'mean'),
    Clientes=('customername', 'nunique')
).round(2).sort_values('Total_USD', ascending=False)
terr.columns = ['N Órdenes', 'Total $', 'Deal Size Prom', 'Descuento % Prom', 'Clientes']
print(tabulate(terr, headers='keys', tablefmt='grid'))


In [ ]:
# ── Función: filtrar por país ─────────────────────────────────────
def filtrar_pais(pais: str):
    """Resumen completo de órdenes de un país."""    subset = df[df['country'].str.contains(pais, case=False, na=False)]
    if subset.empty:
        print(f"❌ No hay órdenes para: {pais}")
        return
    print(f"\n🌎 PAÍS: {pais.upper()}")
    print(f"   N Órdenes       : {len(subset):,}")
    print(f"   Ventas totales  : ${subset['sales'].sum():,.2f}")
    print(f"   Venta promedio  : ${subset['sales'].mean():,.2f}")
    print(f"   Clientes únicos : {subset['customername'].nunique()}")
    print(f"   Líneas          : {', '.join(subset['productline'].unique())}")
    print(f"\n  Top 5 clientes:")
    top = subset.groupby('customername')['sales'].sum().nlargest(5)
    for c, v in top.items():
        print(f"    • {c}: ${v:,.2f}")

# Ejemplo:
filtrar_pais("France")


In [ ]:
# ── Función: filtrar por línea de producto ────────────────────────
def filtrar_producto(linea: str):
    """Resumen completo de una línea de producto."""    subset = df[df['productline'].str.contains(linea, case=False, na=False)]
    if subset.empty:
        print(f"❌ No hay órdenes para línea: {linea}")
        return
    print(f"\n🚗 LÍNEA: {linea.upper()}")
    print(f"   N Órdenes       : {len(subset):,}")
    print(f"   Ventas totales  : ${subset['sales'].sum():,.2f}")
    print(f"   Precio promedio : ${subset['priceeach'].mean():,.2f}")
    print(f"   Descuento prom  : {subset['discount_pct'].mean():.1f}%")
    print(f"   Países activos  : {subset['country'].nunique()}")
    print(f"\n  Top 5 productos:")
    top = subset.groupby('productcode')['sales'].sum().nlargest(5)
    for c, v in top.items():
        print(f"    • {c}: ${v:,.2f}")

# Ejemplo:
filtrar_producto("Classic Cars")


## 📈 PASO 10 — Visualizaciones

In [ ]:
# ── Gráfico 1: Ventas totales por año ────────────────────────────
yr_plot = df.groupby('order_year')['sales'].sum().reset_index()
yr_plot['order_year'] = yr_plot['order_year'].astype(str)

fig, ax = plt.subplots()
bars = ax.bar(yr_plot['order_year'], yr_plot['sales'] / 1e6,
              color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.5)
ax.set_title('Ventas Totales por Año', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Año')
ax.set_ylabel('Ventas (millones $)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fM'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'${bar.get_height():.2f}M', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 2: Ventas por línea de producto ───────────────────────
prod_plot = df.groupby('productline')['sales'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(prod_plot.index, prod_plot.values / 1e6,
               color=sns.color_palette('muted', len(prod_plot)), edgecolor='white')
ax.set_title('Ventas Totales por Línea de Producto', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ventas (millones $)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fM'))
for bar in bars:
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.2f}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 3: Top 10 países por ventas ──────────────────────────
top10_paises = df.groupby('country')['sales'].sum().nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Blues_r', len(top10_paises))
bars = ax.barh(top10_paises.index, top10_paises.values / 1e6,
               color=colors, edgecolor='white')
ax.set_title('Top 10 Países por Ventas Totales', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ventas (millones $)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fM'))
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.2f}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 4: Ventas mensuales por año ──────────────────────────
monthly = df.groupby(['order_year', 'order_month'])['sales'].sum().reset_index()
monthly['order_year'] = monthly['order_year'].astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
for year, group in monthly.groupby('order_year'):
    ax.plot(group['order_month'], group['sales'] / 1e3,
            marker='o', linewidth=2, label=year)
ax.set_title('Ventas Mensuales por Año', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas (miles $)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                    'Jul','Ago','Sep','Oct','Nov','Dic'])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fK'))
ax.legend(title='Año')
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 5: Distribución de deal size ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
deal_counts = df['dealsize'].value_counts()
colors_pie = ['#4C72B0', '#DD8452', '#55A868']
axes[0].pie(deal_counts.values, labels=deal_counts.index,
            autopct='%1.1f%%', colors=colors_pie,
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Distribución de Deal Size', fontsize=13, fontweight='bold')

# Barras por línea de producto
deal_prod = df.groupby(['productline', 'dealsize'])['sales'].sum().unstack(fill_value=0)
deal_prod.plot(kind='bar', ax=axes[1], color=colors_pie, edgecolor='white')
axes[1].set_title('Deal Size por Línea de Producto', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Ventas $')
axes[1].tick_params(axis='x', rotation=30)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].legend(title='Deal Size')

plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 6: Heatmap ventas por mes y año ───────────────────────
pivot_heat = df.pivot_table(
    values='sales', index='order_month',
    columns='order_year', aggfunc='sum', fill_value=0
)
pivot_heat.index = ['Ene','Feb','Mar','Abr','May','Jun',
                    'Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot_heat / 1e3, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Miles $'})
ax.set_title('Heatmap de Ventas por Mes y Año (en miles $)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Año')
ax.set_ylabel('Mes')
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 7: Top 10 clientes ────────────────────────────────────
top10_cli = df.groupby('customername')['sales'].sum().nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top10_cli.index, top10_cli.values / 1e3,
               color=sns.color_palette('Greens_r', 10), edgecolor='white')
ax.set_title('Top 10 Clientes por Ventas Totales', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ventas (miles $)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fK'))
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.1f}K', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 🔎 APÉNDICE — Consultas Rápidas de Referencia

Celdas individuales para responder preguntas específicas sobre el dataset.


In [ ]:
# ── Top 10 órdenes por monto de venta ────────────────────────────
print("🏆 TOP 10 ÓRDENES POR MONTO")
top10_ord = df.nlargest(10, 'sales')[
    ['ordernumber','orderdate','customername','productline',
     'productcode','sales','status','city','country']
].reset_index(drop=True)
display(top10_ord)


In [ ]:
# ── Órdenes canceladas ────────────────────────────────────────────
print("❌ ÓRDENES CANCELADAS")
canceladas = df[df['status'] == 'Cancelled'][
    ['ordernumber','orderdate','customername','productline','sales','country']
].sort_values('sales', ascending=False)
print(f"   Total canceladas : {len(canceladas)}")
print(f"   Monto total      : ${canceladas['sales'].sum():,.2f}")
display(canceladas)


In [ ]:
# ── Productos únicos → ciudades donde se vendieron ───────────────
print("📍 CADA PRODUCTO → CIUDADES DE VENTA")
prod_ciudades = (
    df.groupby(['productcode', 'productline'])['city']
    .agg(lambda x: ', '.join(sorted(x.unique()[:5])))
    .reset_index()
    .rename(columns={'city': 'Ciudades'})
)
display(prod_ciudades.head(20))


In [ ]:
# ── Resumen general del dataset ──────────────────────────────────
print("📈 RESUMEN GENERAL DEL DATASET")
print(f"  Filas             : {len(df):,}")
print(f"  Columnas          : {len(df.columns)}")
print(f"  Ventas totales    : ${df['sales'].sum():,.2f}")
print(f"  Venta máxima      : ${df['sales'].max():,.2f}")
print(f"  Venta mínima      : ${df['sales'].min():,.2f}")
print(f"  Venta promedio    : ${df['sales'].mean():,.2f}")
print(f"  Descuento prom.   : {df['discount_pct'].mean():.1f}%")
print(f"  Años              : {sorted(df['order_year'].dropna().astype(float).astype(int).unique().tolist())}")
print(f"  Países            : {df['country'].nunique()} únicos")
print(f"  Territorios       : {', '.join(df['territory'].dropna().unique())}")
print(f"  Clientes          : {df['customername'].nunique()} únicos")
print(f"  Líneas producto   : {', '.join(df['productline'].unique())}")
print(f"  Deal sizes        : {df['dealsize'].value_counts().to_dict()}")
print(f"  Estados órdenes   : {df['status'].value_counts().to_dict()}")
